# Check `context_window` vs `max_input_tokens` in Google Gemini YAMLs

This notebook scans all `.yaml` files under `models/providers/google-gemini/` and reports any file where `context_window` and `max_input_tokens` are both present but differ.

In [2]:
import yaml
from pathlib import Path

In [3]:
GOOGLE_GEMINI_DIR = Path("../providers/google-gemini/")  # relative to models/local/, pointing at google-gemini/

yaml_files = sorted(GOOGLE_GEMINI_DIR.glob("*.yaml"))
print(f"Found {len(yaml_files)} YAML files")

Found 55 YAML files


In [4]:
results = []

for yaml_file in yaml_files:
    with yaml_file.open() as f:
        data = yaml.safe_load(f)

    if not isinstance(data, dict):
        continue

    limits = data.get("limits", {})
    if not isinstance(limits, dict):
        continue

    context_window = limits.get("context_window")
    max_input_tokens = limits.get("max_input_tokens")

    # Only flag rows where both fields exist and they differ
    if context_window is not None and max_input_tokens is not None:
        if context_window != max_input_tokens:
            results.append({
                "file": yaml_file.name,
                "context_window": context_window,
                "max_input_tokens": max_input_tokens,
                "difference": context_window - max_input_tokens,
            })

print(f"Files with mismatched context_window vs max_input_tokens: {len(results)}")

Files with mismatched context_window vs max_input_tokens: 0


In [4]:
if results:
    print(f"{'File':<55} {'context_window':>16} {'max_input_tokens':>18} {'difference':>12}")
    print("-" * 105)
    for row in results:
        print(f"{row['file']:<55} {row['context_window']:>16,} {row['max_input_tokens']:>18,} {row['difference']:>12,}")
else:
    print("No mismatches found — context_window equals max_input_tokens in all files that define both.")

No mismatches found — context_window equals max_input_tokens in all files that define both.


## Summary of all files and their limit fields

The cell below prints every YAML file alongside whatever limit-related values it exposes, so you can see the full picture at a glance.

In [ ]:
print(f"{'File':<55} {'context_window':>16} {'max_input_tokens':>18}")
print("-" * 93)

for yaml_file in yaml_files:
    with yaml_file.open() as f:
        data = yaml.safe_load(f)

    limits = data.get("limits", {}) if isinstance(data, dict) else {}
    context_window = limits.get("context_window", "-") if isinstance(limits, dict) else "-"
    max_input_tokens = limits.get("max_input_tokens", "-") if isinstance(limits, dict) else "-"

    cw_str = f"{context_window:,}" if isinstance(context_window, int) else context_window
    mit_str = f"{max_input_tokens:,}" if isinstance(max_input_tokens, int) else max_input_tokens

    mismatch_flag = " <-- MISMATCH" if (
        isinstance(context_window, int)
        and isinstance(max_input_tokens, int)
        and context_window != max_input_tokens
    ) else ""

    print(f"{yaml_file.name:<55} {cw_str:>16} {mit_str:>18}{mismatch_flag}")

## Backfill `limits.context_window`

For every YAML that has `limits.max_input_tokens` but is missing `limits.context_window`, the cell below writes `context_window` with the same value as `max_input_tokens`.

In [5]:
patched = []
skipped_deprecated = []
skipped_no_max = []
already_has_cw = []

for yaml_file in yaml_files:
    raw_text = yaml_file.read_text()
    data = yaml.safe_load(raw_text)

    if not isinstance(data, dict):
        continue

    # Skip deprecated models
    if data.get("isDeprecated") is True:
        skipped_deprecated.append(yaml_file.name)
        continue

    limits = data.get("limits")

    # Skip files that have no limits section or no max_input_tokens
    if not isinstance(limits, dict) or "max_input_tokens" not in limits:
        skipped_no_max.append(yaml_file.name)
        continue

    # Skip files that already define context_window
    if "context_window" in limits:
        already_has_cw.append(yaml_file.name)
        continue

    # Set context_window = max_input_tokens and write back
    limits["context_window"] = limits["max_input_tokens"]
    yaml_file.write_text(yaml.dump(data, allow_unicode=True, sort_keys=True))
    patched.append(yaml_file.name)

print(f"Patched ({len(patched)} files):")
for name in patched:
    print(f"  + {name}")

print(f"\nAlready had context_window ({len(already_has_cw)} files):")
for name in already_has_cw:
    print(f"  = {name}")

print(f"\nSkipped — deprecated ({len(skipped_deprecated)} files):")
for name in skipped_deprecated:
    print(f"  ~ {name}")

print(f"\nSkipped — no max_input_tokens ({len(skipped_no_max)} files):")
for name in skipped_no_max:
    print(f"  - {name}")

Patched (25 files):
  + aqa.yaml
  + gemini-2.5-computer-use-preview-10-2025.yaml
  + gemini-2.5-flash-image.yaml
  + gemini-2.5-flash-lite.yaml
  + gemini-2.5-flash-native-audio-latest.yaml
  + gemini-2.5-flash-native-audio-preview-09-2025.yaml
  + gemini-2.5-flash-native-audio-preview-12-2025.yaml
  + gemini-2.5-flash-preview-tts.yaml
  + gemini-2.5-pro-preview-tts.yaml
  + gemini-3-flash-preview.yaml
  + gemini-3-pro-image-preview.yaml
  + gemini-3.1-flash-image-preview.yaml
  + gemini-3.1-flash-lite-preview.yaml
  + gemini-3.1-pro-preview-customtools.yaml
  + gemini-3.1-pro-preview.yaml
  + gemini-embedding-2-preview.yaml
  + gemini-flash-latest.yaml
  + gemini-flash-lite-latest.yaml
  + gemini-robotics-er-1.5-preview.yaml
  + gemma-3-12b-it.yaml
  + gemma-3-27b-it.yaml
  + imagen-4.0-generate-001.yaml
  + nano-banana-pro-preview.yaml
  + veo-3.1-fast-generate-preview.yaml
  + veo-3.1-generate-preview.yaml

Already had context_window (5 files):
  = deep-research-pro-preview-12-2025

## Files with `context_window` but no `max_input_tokens`

In [6]:
cw_only = []

for yaml_file in yaml_files:
    data = yaml.safe_load(yaml_file.read_text())

    if not isinstance(data, dict):
        continue

    limits = data.get("limits")
    if not isinstance(limits, dict):
        continue

    has_context_window = "context_window" in limits
    has_max_input_tokens = "max_input_tokens" in limits

    if has_context_window and not has_max_input_tokens:
        cw_only.append({
            "file": yaml_file.name,
            "context_window": limits["context_window"],
        })

if cw_only:
    print(f"Found {len(cw_only)} file(s) with context_window but no max_input_tokens:\n")
    print(f"{'File':<55} {'context_window':>16}")
    print("-" * 73)
    for row in cw_only:
        print(f"{row['file']:<55} {row['context_window']:>16,}")
else:
    print("No files found with context_window but missing max_input_tokens.")

No files found with context_window but missing max_input_tokens.
